# 🌾 Agriculture de précision — Détection d'anomalies parcellaires
## Phase 1 : Ingestion, Fusion et Sécurisation (RGPD)

---

### Objectif de cette phase

1. **Charger** les deux fichiers CSV dans des DataFrames pandas
2. **Identifier et supprimer** les données personnelles pour se conformer au RGPD
3. **Fusionner** les observations avec les parcelles (Left Join sur `ParcelleID`)
4. **Explorer** le dataset fusionné pour amorcer la Phase 2 (nettoyage)

## 1. Importation des librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration d'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

# Style des graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Chemin vers les données
DATA_DIR = Path('../data')

print('✅ Librairies importées avec succès.')

## 2. Chargement des données

In [ ]:
# Chargement des deux fichiers CSV
df_parcelles = pd.read_csv(DATA_DIR / 'parcelles.csv')
df_observations = pd.read_csv(DATA_DIR / 'observations.csv')

print(f'📋 Parcelles : {df_parcelles.shape[0]:,} lignes × {df_parcelles.shape[1]} colonnes')
print(f'📋 Observations : {df_observations.shape[0]:,} lignes × {df_observations.shape[1]} colonnes')

In [ ]:
# Aperçu du fichier parcelles
print('=== PARCELLES.CSV ===')
display(df_parcelles.head(10))

# Affichage des types avec des noms clairs
mapping_types = {
    'object': 'chaîne de caractères',
    'float64': 'nombre réel',
    'int64': 'nombre entier',
    'datetime64[ns]': 'date et heure'
}
types_clairs = df_parcelles.dtypes.map(lambda t: mapping_types.get(str(t), str(t)))
print()
print('Types de données :')
display(types_clairs.to_frame('Type'))

In [ ]:
# Aperçu du fichier observations
print('=== OBSERVATIONS.CSV ===')
display(df_observations.head(10))

# Réutilisation du mapping_types défini plus haut
types_clairs = df_observations.dtypes.map(lambda t: mapping_types.get(str(t), str(t)))
print()
print('Types de données :')
display(types_clairs.to_frame('Type'))

## 3. Analyse RGPD — Identification des données personnelles

Le RGPD (Règlement Général sur la Protection des Données) impose le principe de **minimisation des données** : seules les données strictement nécessaires à la finalité du traitement doivent être conservées.

### Colonnes identifiées comme données personnelles dans `parcelles.csv` :

| Colonne | Nature | Risque | Décision |
|---------|--------|--------|----------|
| `Exploitant` | Nom et prénom | Identification directe | ❌ Suppression |
| `Email` | Adresse électronique | Contact direct | ❌ Suppression |
| `Telephone` | Numéro de téléphone | Contact direct | ❌ Suppression |
| `NumSIRET` | Identifiant d'exploitation | Identification indirecte (rattachement à une entité juridique) | ❌ Suppression |
| `CodePostal` | Code postal | Localisation approximative | ⚠️ Conservé (nécessaire pour l'analyse régionale) mais nettoyé des valeurs aberrantes |

In [ ]:
# Vérification de l'existence des colonnes avant suppression
colonnes_personnelles = ['Exploitant', 'Email', 'Telephone', 'NumSIRET']
colonnes_existantes = [col for col in colonnes_personnelles if col in df_parcelles.columns]
colonnes_absentes = [col for col in colonnes_personnelles if col not in df_parcelles.columns]

print(f'Colonnes à supprimer trouvées : {colonnes_existantes}')
if colonnes_absentes:
    print(f'Colonnes déjà absentes : {colonnes_absentes}')

# Suppression des colonnes personnelles (principe de minimisation RGPD)
df_parcelles_rgpd = df_parcelles.drop(columns=colonnes_existantes)

print(f'\n✅ Données personnelles supprimées.')
print(f'Avant : {df_parcelles.shape[1]} colonnes → Après : {df_parcelles_rgpd.shape[1]} colonnes')
print(f'Colonnes conservées : {list(df_parcelles_rgpd.columns)}')

### Justification de la suppression du NumSIRET

Le `NumSIRET` est un identifiant d'établissement qui permet de remonter à l'exploitant via les bases publiques (INSEE, SIRENE). Bien qu'il ne soit pas directement nominatif, il constitue une **donnée indirectement identifiante** au sens du RGPD.

Une alternative aurait été le **hachage** (SHA-256) pour conserver la capacité de regroupement par exploitation sans connaître l'identité. Cependant, dans le cadre de ce projet, la notion d'exploitation n'est pas nécessaire au modèle : l'unité d'analyse est la **parcelle**, pas l'exploitant.

👉 **Décision : suppression pure et simple.**

## 4. Fusion des données — Left Join

Nous effectuons un **Left Join** des observations (table principale) avec les parcelles (table secondaire) sur la clé `ParcelleID`.

**Pourquoi un Left Join ?**
- Chaque observation est l'unité d'analyse (une prédiction = une observation)
- On souhaite enrichir chaque observation avec les caractéristiques de sa parcelle
- Un Left Join garantit qu'aucune observation n'est perdue, même si une parcelle venait à manquer

In [ ]:
# Vérification des clés de jointure
print(f'Nb parcelles uniques dans parcelles.csv : {df_parcelles_rgpd["ParcelleID"].nunique()}')
print(f'Nb parcelles uniques dans observations.csv : {df_observations["ParcelleID"].nunique()}')

# Identification des parcelles présentes dans observations mais absentes de parcelles
parcelles_obs = set(df_observations['ParcelleID'].unique())
parcelles_ref = set(df_parcelles_rgpd['ParcelleID'].unique())
orphelines = parcelles_obs - parcelles_ref

if orphelines:
    print()
    print(f'Parcelles dans observations mais ABSENTES de parcelles : {len(orphelines)}')
    print('-' * 60)
    for pid in sorted(orphelines):
        nb_obs = (df_observations['ParcelleID'] == pid).sum()
        print(f'  {pid}  →  {nb_obs} observation(s) concernée(s)')
    print()
    print('Ces observations seront conservées mais les colonnes "parcelles"')
    print('seront vides (NaN) après le Left Join.')
else:
    print('Toutes les parcelles des observations existent dans le fichier parcelles.')

print()
print('Parcelles dans parcelles mais JAMAIS observées :')
parcelles_non_observees = parcelles_ref - parcelles_obs
print(f'  {len(parcelles_non_observees)} parcelles sans aucune observation.')


In [ ]:
# Fusion Left Join : observations ← parcelles
df = df_observations.merge(df_parcelles_rgpd, on='ParcelleID', how='left')

print(f'📊 Dataset fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'\nColonnes du dataset fusionné :')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

## 5. Exploration initiale du dataset fusionné

In [ ]:
# Informations générales sur le dataset
# On évite df.info() qui affiche des termes techniques comme "object" ou "float64".
# À la place, on construit un tableau clair avec des types compréhensibles.

mapping_types = {
    'object': 'texte',
    'float64': 'réel',
    'int64': 'entier',
    'datetime64[ns]': 'date'
}

resume = pd.DataFrame({
    'Colonne': df.columns,
    'Nb Non Null': df.notna().sum().values,
    'Nb Null': df.isna().sum().values,
    'Type': df.dtypes.map(lambda t: mapping_types.get(str(t), str(t)))
})

# Nombre total de lignes
print(f'Nombre de lignes  : {len(df):,}')
print(f'Nombre de colonnes : {len(df.columns)}')
print()
display(resume)

In [ ]:
# Statistiques descriptives des variables numériques
df.describe()

## 5. Visualisation des valeurs manquantes

Avant toute modélisation, il est essentiel de quantifier et visualiser les données absentes. 
Elles peuvent indiquer des **capteurs défaillants**, des **données non disponibles à certains stades de culture**, ou des **erreurs de saisie**.

In [ ]:
# Visualisation des valeurs manquantes
import missingno as msno

# Barplot : nombre de valeurs manquantes par colonne
fig, ax = plt.subplots(figsize=(14, 6))
msno.bar(df, ax=ax, color=(0.2, 0.5, 0.8), fontsize=10)
ax.set_title('Valeurs manquantes par colonne', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Tableau numérique pour compléter le graphique
print('=== NOMBRE DE VALEURS MANQUANTES PAR COLONNE ===')
manquants = df.isna().sum()
manquants_pct = (df.isna().sum() / len(df) * 100).round(2)
summary_na = pd.DataFrame({
    'Nb manquants': manquants,
    '% manquants': manquants_pct
})
summary_na = summary_na[summary_na['Nb manquants'] > 0].sort_values('Nb manquants', ascending=False)
display(summary_na)

### 🟢 Interprétation des valeurs manquantes

Le barplot et le tableau ci-dessus montrent la répartition des données absentes :

1. **NDVI** (307 manquants, ~3,1 %) — Certains relevés drone ou satellite peuvent être inexploitables (couverture nuageuse, angle de prise de vue). C'est la colonne la plus touchée.

2. **RendementEstime_t_ha** (231 manquants, ~2,3 %) — L'estimation de rendement n'est pas disponible à tous les stades de culture (difficile d'estimer un rendement en plein semis ou levée).

3. **Temperature, Pluviometrie_mm et Humidite** (~150–220 manquants) — Probablement des pannes ponctuelles de stations météo ou des défauts de transmission. L'ordre de grandeur reste faible (< 2,5 %).

4. **Capteur, AnomalieLabel, ParcelleID, StadeCulture** — Aucune valeur manquante, ce qui est rassurant pour la cible et les clés.

5. **RendementMoyenZone_t_ha** — Aucune valeur manquante, logique puisque c'est une moyenne historique toujours disponible.

👉 Ces valeurs seront traitées dans la **Phase 2** (imputation par médiane régionale, médiane par culture, ou mode selon la variable).

## 6. Distribution de la variable cible — `AnomalieLabel`

In [ ]:
# Analyse de la cible
anomalie_counts = df['AnomalieLabel'].value_counts()
anomalie_pct = (df['AnomalieLabel'].value_counts(normalize=True) * 100).round(2)

print('=== DISTRIBUTION DE LA CIBLE ===')
for label in [0, 1]:
    libelle = 'Normal' if label == 0 else 'Anomalie'
    print(f'{libelle} (Label={label}) : {anomalie_counts[label]:,} observations ({anomalie_pct[label]}%)')

In [ ]:
# Graphique de la distribution de la cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(['Normal (0)', 'Anomalie (1)'], anomalie_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel("Nombre d'observations", fontsize=12)
axes[0].set_title('Distribution de la variable cible AnomalieLabel', fontsize=13, fontweight='bold')
for bar, count, pct in zip(bars, anomalie_counts.values, anomalie_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                 f'{count:,}\n({pct}%)', ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(anomalie_counts.values, labels=['Normal (0)', 'Anomalie (1)'], 
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12})
axes[1].set_title('Proportion Normal vs Anomalie', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 🟢 Interprétation du déséquilibre de classes

La variable cible `AnomalieLabel` présente un **déséquilibre** entre les deux classes. Cela est attendu dans un contexte agricole : les situations normales sont plus fréquentes que les anomalies.

**Conséquence pour la modélisation :**
- Un modèle naïf qui prédirait toujours « Normal » obtiendrait un bon score d'accuracy mais serait incapable de détecter la moindre anomalie — ce qui est inacceptable pour la coopérative.
- Nous devrons donc utiliser :
  - Des métriques robustes au déséquilibre : **Recall**, **F1-score**, **ROC-AUC** (pas l'accuracy seule)
  - Des techniques de rééquilibrage : **SMOTE** (suréchantillonnage synthétique) ou pondération des classes (`scale_pos_weight` pour XGBoost)

👉 Le traitement du déséquilibre sera détaillé dans la **Phase 4 (Modélisation)**.

## 7. Premières statistiques par groupe (Normal vs Anomalie)

Avant de passer au nettoyage, observons quelques différences entre les observations normales et anormales.

In [ ]:
# Comparaison des moyennes des variables numériques par AnomalieLabel
colonnes_numeriques = ['Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI', 
                       'RendementEstime_t_ha', 'RendementMoyenZone_t_ha', 'Surface_ha']

comparaison = df.groupby('AnomalieLabel')[colonnes_numeriques].mean().round(2)
comparaison.index = ['Normal (0)', 'Anomalie (1)']

print('=== MOYENNES PAR CLASSE ===')
display(comparaison)

In [ ]:
# Sauvegarde du dataset fusionné nettoyé RGPD pour la Phase 2
df.to_csv(DATA_DIR / 'dataset_fusionne_rgpd.csv', index=False)
print('✅ Dataset fusionné sauvegardé : data/dataset_fusionne_rgpd.csv')
print(f'   Dimensions finales : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

## ✅ Bilan de la Phase 1

| Étape | Statut | Détail |
|-------|:------:|--------|
| Chargement des CSV | ✅ | `parcelles.csv` (500 lignes) + `observations.csv` (10 000 lignes) |
| Suppression RGPD | ✅ | `Exploitant`, `Email`, `Telephone`, `NumSIRET` supprimés |
| Fusion Left Join | ✅ | 10 000 observations enrichies avec les caractéristiques des parcelles |
| Exploration initiale | ✅ | Identification des valeurs manquantes et du déséquilibre de classes |
| Sauvegarde | ✅ | Dataset fusionné exporté pour la Phase 2 |

---
# 🧹 Phase 2 : Nettoyage avancé et Analyse Exploratoire (EDA)

### Objectif de cette phase

1. **Détecter et corriger** les valeurs aberrantes dans chaque colonne
2. **Imputer** les valeurs manquantes avec des stratégies adaptées
3. **Visualiser** les distributions et relations entre variables
4. **Identifier** les premiers signaux discriminants entre Normal et Anomalie

## 1. Diagnostic des valeurs aberrantes

Avant de nettoyer, nous allons inspecter chaque variable pour identifier les anomalies de plage.

In [ ]:
# Recharger le dataset fusionné (si on reprend ici)
# df = pd.read_csv(DATA_DIR / 'dataset_fusionne_rgpd.csv')

print(f'Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

In [ ]:
# Fonction de diagnostic : min, max et bornes attendues pour chaque variable
def diagnostiquer_variable(df, colonne, min_attendu, max_attendu):
    val_min = df[colonne].min()
    val_max = df[colonne].max()
    nb_manquantes = df[colonne].isna().sum()
    nb_hors_bornes = df[(df[colonne] < min_attendu) | (df[colonne] > max_attendu)][colonne].count()
    
    statut = '✅' if nb_hors_bornes == 0 else f'⚠️ {nb_hors_bornes} valeurs hors bornes'
    
    print(f'{colonne:<30} | Min={val_min:>8.2f} | Max={val_max:>8.2f} | '
          f'Attendu=[{min_attendu}, {max_attendu}] | NaN={nb_manquantes} | {statut}')

print('=== DIAGNOSTIC DES VALEURS ABERRANTES ===')
print('')
diagnostiquer_variable(df, 'Temperature', -20, 55)
diagnostiquer_variable(df, 'Humidite', 0, 100)
diagnostiquer_variable(df, 'Pluviometrie_mm', 0, 500)
diagnostiquer_variable(df, 'NDVI', -1, 1)
diagnostiquer_variable(df, 'Surface_ha', 0, 1000)
diagnostiquer_variable(df, 'RendementEstime_t_ha', 0, 20)
diagnostiquer_variable(df, 'RendementMoyenZone_t_ha', 0, 20)

In [ ]:
# Détail des surfaces négatives
surfaces_negatives = df[df['Surface_ha'] < 0]
print(f'⚠️ Surfaces négatives détectées : {len(surfaces_negatives)} parcelles')
if len(surfaces_negatives) > 0:
    display(surfaces_negatives[['ParcelleID', 'Surface_ha']].drop_duplicates())

In [ ]:
# Détail des CodePostal invalides (XXXXX)
codes_invalides = df[df['CodePostal'].astype(str).str.contains(r'[A-Za-z]', na=False)]
print(f'⚠️ Codes postaux avec caractères non numériques : {codes_invalides["ParcelleID"].nunique()} parcelles')
if len(codes_invalides) > 0:
    display(codes_invalides[['ParcelleID', 'CodePostal']].drop_duplicates())

In [ ]:
# Visualisation : Boxplots avant nettoyage
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

variables = ['Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI', 'Surface_ha', 'RendementEstime_t_ha']
titres = ['Température (°C)', 'Humidité (%)', 'Pluviométrie (mm)', 'NDVI', 'Surface (ha)', 'Rendement estimé (t/ha)']

for ax, var, titre in zip(axes.flat, variables, titres):
    ax.boxplot(df[var].dropna(), vert=True, patch_artist=True,
              boxprops=dict(facecolor='#3498db', alpha=0.7),
              flierprops=dict(marker='o', markerfacecolor='red', markersize=4))
    ax.set_title(titre, fontsize=12, fontweight='bold')
    ax.set_ylabel('')

fig.suptitle('Boxplots avant nettoyage — Détection des valeurs aberrantes', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🟢 Interprétation des boxplots avant nettoyage

Les points rouges (outliers) révèlent plusieurs anomalies :
- **Température** : des valeurs extrêmes bien au-delà des 55°C possibles en agriculture (probablement des erreurs de capteur)
- **Humidité** : des pourcentages dépassant 100% — physiquement impossibles
- **Pluviométrie** : des valeurs négatives — absurdes (une précipitation ne peut pas être négative)
- **NDVI** : des valeurs bien en dehors de l'intervalle théorique [-1, 1]
- **Surface** : des hectares négatifs (erreur de saisie probable)
- **Rendement estimé** : la distribution semble globalement cohérente

👉 Ces valeurs aberrantes doivent être traitées avant toute modélisation, car elles fausseraient l'apprentissage.

## 2. Nettoyage des valeurs aberrantes

In [ ]:
# Copie du dataframe pour le nettoyage
df_clean = df.copy()

# Compteur de corrections
corrections = {}

# --- 1. Surface_ha : remplacer les valeurs négatives par leur valeur absolue ---
neg_surface = (df_clean['Surface_ha'] < 0).sum()
df_clean['Surface_ha'] = df_clean['Surface_ha'].abs()
corrections['Surface_ha (negatives -> abs)'] = neg_surface

# --- 2. CodePostal : remplacer 'XXXXX' par NaN ---
mask_cp_invalide = df_clean['CodePostal'].astype(str).str.contains(r'[A-Za-z]', na=False)
nb_cp_invalide = mask_cp_invalide.sum()
df_clean.loc[mask_cp_invalide, 'CodePostal'] = np.nan
corrections['CodePostal (XXXXX -> NaN)'] = nb_cp_invalide

# --- 3. Temperature : borner à [-20, 55] ---
temp_basse = (df_clean['Temperature'] < -20).sum()
temp_haute = (df_clean['Temperature'] > 55).sum()
df_clean.loc[df_clean['Temperature'] < -20, 'Temperature'] = np.nan
df_clean.loc[df_clean['Temperature'] > 55, 'Temperature'] = np.nan
corrections['Temperature (< -20 -> NaN)'] = temp_basse
corrections['Temperature (> 55 -> NaN)'] = temp_haute

# --- 4. Humidite : borner à [0, 100] ---
hum_basse = (df_clean['Humidite'] < 0).sum()
hum_haute = (df_clean['Humidite'] > 100).sum()
df_clean.loc[df_clean['Humidite'] < 0, 'Humidite'] = np.nan
df_clean.loc[df_clean['Humidite'] > 100, 'Humidite'] = np.nan
corrections['Humidite (< 0 -> NaN)'] = hum_basse
corrections['Humidite (> 100 -> NaN)'] = hum_haute

# --- 5. Pluviometrie_mm : borner à >= 0 ---
pluv_neg = (df_clean['Pluviometrie_mm'] < 0).sum()
df_clean.loc[df_clean['Pluviometrie_mm'] < 0, 'Pluviometrie_mm'] = np.nan
corrections['Pluviometrie (< 0 -> NaN)'] = pluv_neg

# --- 6. NDVI : borner à [-1, 1] ---
ndvi_bas = (df_clean['NDVI'] < -1).sum()
ndvi_haut = (df_clean['NDVI'] > 1).sum()
df_clean.loc[df_clean['NDVI'] < -1, 'NDVI'] = np.nan
df_clean.loc[df_clean['NDVI'] > 1, 'NDVI'] = np.nan
corrections['NDVI (< -1 -> NaN)'] = ndvi_bas
corrections['NDVI (> 1 -> NaN)'] = ndvi_haut

# --- 7. RendementEstime_t_ha : borner à des valeurs physiquement plausibles [0, 20] ---
rend_neg = (df_clean['RendementEstime_t_ha'] < 0).sum()
rend_haute = (df_clean['RendementEstime_t_ha'] > 20).sum()
df_clean.loc[df_clean['RendementEstime_t_ha'] < 0, 'RendementEstime_t_ha'] = np.nan
df_clean.loc[df_clean['RendementEstime_t_ha'] > 20, 'RendementEstime_t_ha'] = np.nan
corrections['Rendement (< 0 -> NaN)'] = rend_neg
corrections['Rendement (> 20 -> NaN)'] = rend_haute

# Bilan des corrections
print('=== BILAN DES CORRECTIONS ===')
total = 0
for action, nb in corrections.items():
    if nb > 0:
        print(f'  {action:<45} : {nb:>5} valeurs')
        total += nb
print(f'  {"TOTAL":<45} : {total:>5} valeurs corrigées')

In [ ]:
# Visualisation des boxplots après nettoyage
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, var, titre in zip(axes.flat, variables, titres):
    ax.boxplot(df_clean[var].dropna(), vert=True, patch_artist=True,
              boxprops=dict(facecolor='#2ecc71', alpha=0.7),
              flierprops=dict(marker='o', markerfacecolor='orange', markersize=4))
    ax.set_title(f'{titre} (après nettoyage)', fontsize=12, fontweight='bold')

fig.suptitle('Boxplots après nettoyage — Valeurs aberrantes traitées', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🟢 Interprétation après nettoyage

Les boxplots après nettoyage montrent des distributions bien plus cohérentes :
- Les valeurs extrêmes aberrantes ont disparu
- Les points orange restants sont des outliers statistiques mais **physiquement plausibles** (ex: une température à 40°C en été, un NDVI très bas sur un sol nu)
- Ces outliers légitimes seront conservés : ils peuvent être porteurs d'information pour la détection d'anomalies

## 3. Imputation des valeurs manquantes

Avant d'imputer, il faut choisir une stratégie adaptée à chaque variable. Voici le raisonnement :

### Stratégie par variable

| Variable | Méthode | Justification |
|----------|---------|---------------|
| Temperature, Humidite, Pluviometrie_mm, NDVI | **Médiane par Région** | Les conditions météo varient selon la géographie. Imputer avec la médiane de la même région est plus pertinent qu'une médiane nationale. Exemple : une température manquante en Bretagne sera remplacée par la température médiane des autres relevés bretons. |
| RendementEstime_t_ha, RendementMoyenZone_t_ha | **Médiane par TypeCulture** | Le rendement dépend fortement de la culture (le blé ne produit pas comme la vigne). Imputer avec la médiane du même type de culture évite des aberrations. Exemple : un rendement manquant sur du Maïs sera remplacé par le rendement médian de tous les autres relevés Maïs. |
| Capteur | **Mode global** | Peu de valeurs manquantes, on prend le type de capteur le plus fréquent tout court. |
| CodePostal | **Mode par Région** | Le code postal est lié à la géographie. Imputer avec le code postal le plus fréquent **de la même Région** garantit une cohérence territoriale. Exemple : un code postal manquant en Nouvelle-Aquitaine sera remplacé par le code postal le plus courant dans cette région (probablement autour de Bordeaux), et non par un code postal parisien. |

### Pourquoi la médiane plutôt que la moyenne ?

La médiane est **robuste aux valeurs extrêmes**. Même après le bornage de la Phase 2, il peut rester des valeurs atypiques. Une moyenne serait tirée vers le haut par un pic de température à 45°C, alors que la médiane reste représentative du cas « normal ».

### Fallback

Si après imputation par groupe il reste des NaN (cas rare : une région entière sans donnée pour une variable), on applique la médiane/mode **globale**.

In [ ]:
# État des valeurs manquantes avant imputation
nan_avant = df_clean.isna().sum()
print('=== VALEURS MANQUANTES AVANT IMPUTATION ===')
display(nan_avant[nan_avant > 0].to_frame('Nb NaN'))

In [ ]:
# Imputation des variables météorologiques par la médiane régionale
vars_meteo = ['Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI']

for var in vars_meteo:
    # Calculer la médiane par Région
    mediane_region = df_clean.groupby('Region')[var].transform('median')
    # Remplir les NaN
    df_clean[var] = df_clean[var].fillna(mediane_region)
    # Si encore des NaN (région entière sans données), utiliser la médiane globale
    if df_clean[var].isna().any():
        df_clean[var] = df_clean[var].fillna(df_clean[var].median())

print('✅ Variables météo imputées (médiane par Région)')

In [ ]:
# Imputation des rendements par la médiane par TypeCulture
vars_rendement = ['RendementEstime_t_ha', 'RendementMoyenZone_t_ha']

for var in vars_rendement:
    mediane_culture = df_clean.groupby('TypeCulture')[var].transform('median')
    df_clean[var] = df_clean[var].fillna(mediane_culture)
    # Fallback médiane globale
    if df_clean[var].isna().any():
        df_clean[var] = df_clean[var].fillna(df_clean[var].median())

print('✅ Rendements imputés (médiane par TypeCulture)')

In [ ]:
# Imputation du Capteur par le mode (valeur la plus fréquente)
mode_capteur = df_clean['Capteur'].mode()[0]
nb_nan_capteur = df_clean['Capteur'].isna().sum()
df_clean['Capteur'] = df_clean['Capteur'].fillna(mode_capteur)
print(f'✅ Capteur : {nb_nan_capteur} valeurs manquantes imputées par le mode : "{mode_capteur}"')

In [ ]:
# Imputation du CodePostal par le mode par Région
mode_cp_region = df_clean.groupby('Region')['CodePostal'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else '00000')
nb_nan_cp = df_clean['CodePostal'].isna().sum()
df_clean['CodePostal'] = df_clean['CodePostal'].fillna(mode_cp_region)
print(f'✅ CodePostal : {nb_nan_cp} valeurs manquantes imputées par le mode régional')

In [ ]:
# Vérification finale : plus aucune valeur manquante
nan_apres = df_clean.isna().sum().sum()
print(f'\n=== VÉRIFICATION FINALE ===')
print(f'Valeurs manquantes restantes : {nan_apres}')
if nan_apres == 0:
    print('✅ Dataset parfaitement nettoyé — aucune valeur manquante !')
else:
    print(f'⚠️ Il reste {nan_apres} valeurs manquantes.')
    display(df_clean.isna().sum()[df_clean.isna().sum() > 0].to_frame('Nb NaN'))

## 4. Analyse Exploratoire (EDA) — Distributions par AnomalieLabel

Comparons la distribution des variables numériques entre les observations normales et les anomalies.

In [ ]:
# Distribution des variables numériques par classe
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, var, titre in zip(axes.flat, variables, titres):
    for label, color, nom in [(0, '#2ecc71', 'Normal'), (1, '#e74c3c', 'Anomalie')]:
        data = df_clean[df_clean['AnomalieLabel'] == label][var]
        ax.hist(data, bins=40, alpha=0.6, color=color, label=nom, density=True)
    ax.set_title(titre, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Distribution des variables numériques — Normal vs Anomalie', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🟢 Interprétation des distributions comparées

Ces histogrammes superposés permettent d'identifier quelles variables sont les plus discriminantes :
- **NDVI** : On s'attend à ce que les anomalies présentent des NDVI plus faibles (végétation stressée ou malade)
- **Température** : Des températures extrêmes (trop basses ou trop élevées) peuvent être corrélées aux anomalies
- **Humidité** : Un stress hydrique se traduit par une humidité anormale
- **Rendement estimé** : Un écart par rapport au rendement moyen de la zone est un signal fort d'anomalie
- **Surface** : La taille de la parcelle peut influencer la détection (les petites parcelles sont parfois moins bien suivies)

👉 Ces observations guideront le **feature engineering** en Phase 3.

## 5. Analyse des variables catégorielles

In [ ]:
# Distribution des anomalies par TypeCulture
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

var_cat = ['TypeCulture', 'TypeSol', 'Irrigation', 'Region', 'Capteur', 'StadeCulture']

for ax, var in zip(axes.flat, var_cat):
    # Calculer le taux d'anomalie par catégorie
    taux_anomalie = df_clean.groupby(var)['AnomalieLabel'].mean().sort_values(ascending=False) * 100
    couleurs = ['#e74c3c' if x > df_clean['AnomalieLabel'].mean() * 100 else '#3498db' for x in taux_anomalie.values]
    
    bars = ax.barh(range(len(taux_anomalie)), taux_anomalie.values, color=couleurs, edgecolor='white')
    ax.set_yticks(range(len(taux_anomalie)))
    ax.set_yticklabels(taux_anomalie.index, fontsize=9)
    ax.set_xlabel("Taux d'anomalie (%)", fontsize=10)
    ax.set_title(f"Taux d'anomalie par {var}", fontsize=12, fontweight='bold')
    ax.axvline(df_clean['AnomalieLabel'].mean() * 100, color='gray', linestyle='--', alpha=0.7, label='Moyenne globale')
    
    # Ajouter les valeurs
    for bar, val in zip(bars, taux_anomalie.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8)

fig.suptitle("Taux d'anomalie par variable catégorielle", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🟢 Interprétation des taux d'anomalie par catégorie

Ce graphique est crucial pour identifier les **biais potentiels** dans les données :

- **TypeCulture** : Si certaines cultures ont un taux d'anomalie significativement plus élevé que la moyenne, cela peut indiquer soit un vrai risque agronomique, soit un biais de surveillance (plus de capteurs sur cette culture).
- **Région** : Un déséquilibre géographique pourrait signaler que certaines régions sont sur- ou sous-équipées en capteurs.
- **Capteur** : Si les anomalies sont plus souvent détectées par un type de capteur (ex: Drone vs Station_Sol), cela peut révéler un biais instrumental.
- **StadeCulture** : Certains stades (floraison, maturation) sont naturellement plus sensibles aux anomalies.

👉 Ces observations seront documentées dans l'analyse des **risques éthiques (C2)** et prises en compte dans la modélisation.

## 6. Matrice de corrélation

In [ ]:
# Matrice de corrélation des variables numériques
colonnes_num = ['Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI', 
                'RendementEstime_t_ha', 'RendementMoyenZone_t_ha', 
                'Surface_ha', 'AnomalieLabel']

corr_matrix = df_clean[colonnes_num].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Coefficient de corrélation'})
ax.set_title('Matrice de corrélation — Variables numériques', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 🟢 Interprétation de la matrice de corrélation

La dernière ligne (`AnomalieLabel`) indique la corrélation de chaque variable avec notre cible :
- Une **corrélation positive** signifie que la variable tend à augmenter quand il y a une anomalie
- Une **corrélation négative** signifie qu'elle tend à diminuer

Points d'attention :
- Si `NDVI` est négativement corrélé aux anomalies, cela confirme que la baisse de vigueur végétative est un signal fort
- Les fortes corrélations entre variables explicatives (ex: `RendementEstime_t_ha` et `RendementMoyenZone_t_ha`) ne sont pas problématiques pour XGBoost, mais le sont pour la régularisation de la régression logistique
- Si les corrélations avec la cible sont faibles, cela signifie qu'il faudra créer des **features composites** (interactions, ratios) en Phase 3

## 7. Comparaison NDVI / Rendement — Nuage de points

In [ ]:
# Nuage de points : NDVI vs Rendement estimé, coloré par AnomalieLabel
fig, ax = plt.subplots(figsize=(12, 8))

for label, color, nom, marker in [(0, '#2ecc71', 'Normal', 'o'), (1, '#e74c3c', 'Anomalie', 'X')]:
    data = df_clean[df_clean['AnomalieLabel'] == label]
    ax.scatter(data['NDVI'], data['RendementEstime_t_ha'], 
              c=color, label=nom, marker=marker, alpha=0.5, s=30, edgecolors='none')

ax.set_xlabel('NDVI', fontsize=12)
ax.set_ylabel('Rendement estimé (t/ha)', fontsize=12)
ax.set_title('Relation NDVI — Rendement estimé (Normal vs Anomalie)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🟢 Interprétation du nuage NDVI × Rendement

Ce graphique croise deux indicateurs clés de la santé d'une parcelle :
- **NDVI** (vigueur végétative) et **Rendement estimé** (productivité)

On s'attend à ce que :
- Les anomalies (rouge) se concentrent dans les zones de faible NDVI et/ou faible rendement
- Les situations normales (vert) occupent le cadran supérieur droit (bon NDVI + bon rendement)
- Si les deux classes se chevauchent fortement, cela justifie l'utilisation d'un modèle non linéaire (XGBoost) plutôt qu'une simple régression logistique

In [ ]:
# Sauvegarde du dataset nettoyé pour la Phase 3
df_clean.to_csv(DATA_DIR / 'dataset_clean.csv', index=False)
print('✅ Dataset nettoyé sauvegardé : data/dataset_clean.csv')
print(f'   Dimensions : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes')
print(f'   Valeurs manquantes : {df_clean.isna().sum().sum()}')

## ✅ Bilan de la Phase 2

| Étape | Statut | Détail |
|-------|:------:|--------|
| Diagnostic valeurs aberrantes | ✅ | Température, Humidité, Pluviométrie, NDVI, Surface |
| Correction surfaces négatives | ✅ | Passage en valeur absolue |
| Nettoyage CodePostal | ✅ | `XXXXX` → NaN puis imputation mode régional |
| Bornage variables physiques | ✅ | T°, Humidité, Pluvio, NDVI bornés dans leurs intervalles de validité |
| Imputation valeurs manquantes | ✅ | Médiane régionale (météo), médiane par culture (rendement), mode (capteur, CP) |
| EDA — Distributions comparées | ✅ | Histogrammes Normal vs Anomalie pour chaque variable numérique |
| EDA — Analyse catégorielle | ✅ | Taux d'anomalie par TypeCulture, Région, Capteur, StadeCulture... |
| Matrice de corrélation | ✅ | Identification des variables les plus liées à AnomalieLabel |
| Nuage NDVI × Rendement | ✅ | Visualisation du pouvoir discriminant des deux indicateurs phares |
| Sauvegarde | ✅ | `dataset_clean.csv` prêt pour le Feature Engineering |

### 🔜 Prochaine étape : Phase 3 — Feature Engineering

Nous allons créer de nouvelles variables pour enrichir le modèle :
- Âge de la culture (jours depuis la mise en culture)
- Écart au rendement moyen de la zone
- Ratio rendement estimé / rendement moyen
- Encodage One-Hot des variables catégorielles
- Encodage Ordinal du StadeCulture

---
# 📐 Phase 3 : Feature Engineering & Encodage

### Objectif de cette phase

1. **Créer des variables temporelles** : âge de la culture au moment de l'observation
2. **Créer des variables métier** : écarts et ratios de rendement
3. **Encoder les variables catégorielles** : One-Hot Encoding et Ordinal Encoding
4. **Séparer features (X) et cible (y)** avant la modélisation

## 1. Chargement du dataset nettoyé

In [ ]:
# Chargement du dataset issu de la Phase 2
df = pd.read_csv(DATA_DIR / 'dataset_clean.csv')

# Conversion des dates au format datetime
df['DateObservation'] = pd.to_datetime(df['DateObservation'])
df['DateMiseEnCulture'] = pd.to_datetime(df['DateMiseEnCulture'])

print(f'Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Valeurs manquantes : {df.isna().sum().sum()}')
display(df.head(3))

## 2. Feature Engineering — Variables temporelles

In [ ]:
# Calcul de l'âge de la culture (en jours) au moment de l'observation
df['AgeCulture_jours'] = (df['DateObservation'] - df['DateMiseEnCulture']).dt.days

# Vérification
print('=== ÂGE DE LA CULTURE (jours) ===')
print(df['AgeCulture_jours'].describe())
print(f'\nValeurs négatives : {(df["AgeCulture_jours"] < 0).sum()} (observation avant mise en culture)')
print(f'Valeurs nulles     : {(df["AgeCulture_jours"] == 0).sum()}')

# Correction des âges négatifs (erreurs de saisie) → valeur absolue
negatifs = (df['AgeCulture_jours'] < 0).sum()
df['AgeCulture_jours'] = df['AgeCulture_jours'].abs()
if negatifs > 0:
    print(f'\n{negatifs} âges négatifs corrigés (valeur absolue).')

### 🟢 Interprétation de l'âge de culture

L'âge de la culture est une variable clé car le NDVI et le rendement attendu évoluent tout au long du cycle cultural :
- Un NDVI de 0.3 est normal en début de cycle (semis/levée) mais anormal en pleine floraison
- En croisant l'âge avec le stade phénologique, on peut détecter des incohérences (ex: stade « Récolte » à 10 jours)
- Cette variable permet au modèle d'apprendre des patterns saisonniers

## 3. Feature Engineering — Variables métier (écarts de rendement)

In [ ]:
# Écart absolu entre rendement estimé et rendement moyen de la zone
df['EcartRendement_t_ha'] = df['RendementEstime_t_ha'] - df['RendementMoyenZone_t_ha']

# Ratio rendement estimé / rendement moyen (attention à la division par zéro)
df['RatioRendement'] = np.where(
    df['RendementMoyenZone_t_ha'] > 0,
    df['RendementEstime_t_ha'] / df['RendementMoyenZone_t_ha'],
    1.0  # Si le rendement moyen est 0, ratio = 1 (pas d'écart)
)

print('=== ÉCART DE RENDEMENT (t/ha) ===')
print(df['EcartRendement_t_ha'].describe())
print(f'\n=== RATIO DE RENDEMENT ===')
print(df['RatioRendement'].describe())

In [ ]:
# Visualisation : distribution de l'écart de rendement par AnomalieLabel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme de l'écart
for label, color, nom in [(0, '#2ecc71', 'Normal'), (1, '#e74c3c', 'Anomalie')]:
    data = df[df['AnomalieLabel'] == label]['EcartRendement_t_ha']
    axes[0].hist(data, bins=40, alpha=0.6, color=color, label=nom, density=True)
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Écart de rendement (t/ha)', fontsize=11)
axes[0].set_ylabel('Densité', fontsize=11)
axes[0].set_title("Distribution de l'écart de rendement", fontsize=12, fontweight="bold")
axes[0].legend()

# Boxplot de l'écart par AnomalieLabel
df_box = df[['AnomalieLabel', 'EcartRendement_t_ha']].copy()
df_box['Classe'] = df_box['AnomalieLabel'].map({0: 'Normal', 1: 'Anomalie'})
bp = axes[1].boxplot([df_box[df_box['Classe'] == 'Normal']['EcartRendement_t_ha'],
                       df_box[df_box['Classe'] == 'Anomalie']['EcartRendement_t_ha']],
                      labels=['Normal', 'Anomalie'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_ylabel('Écart de rendement (t/ha)', fontsize=11)
axes[1].set_title('Écart de rendement par classe', fontsize=12, fontweight='bold')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 🟢 Interprétation de l'écart de rendement

L'écart entre le rendement estimé et le rendement moyen de la zone est un **signal fort** d'anomalie :
- Un écart **négatif** important (rendement estimé << moyenne zone) suggère un problème sur la parcelle
- Un écart **proche de zéro** est rassurant : la parcelle se comporte comme la moyenne
- Le boxplot montre si les anomalies se distinguent nettement des situations normales sur cet indicateur

👉 Cette variable sera l'une des plus importantes pour le modèle.

## 4. Suppression des colonnes non nécessaires à la modélisation

In [ ]:
# Colonnes à conserver pour la modélisation
# On supprime les identifiants, les dates (déjà transformées) et les colonnes redondantes
colonnes_a_supprimer = [
    'ObservationID',      # Identifiant unique, pas de valeur prédictive
    'ParcelleID',         # Identifiant, pas de valeur prédictive (déjà utilisé pour la fusion)
    'DateObservation',    # Date transformée via AgeCulture_jours
    'DateMiseEnCulture',  # Date transformée via AgeCulture_jours
    'CodePostal'          # Information redondante avec Region (trop granulaire)
]

df_model = df.drop(columns=colonnes_a_supprimer)

print(f'Colonnes supprimées : {colonnes_a_supprimer}')
print(f'Dimensions avant : {df.shape[1]} colonnes → Après : {df_model.shape[1]} colonnes')
print(f'\nColonnes conservées ({df_model.shape[1]}) :')
for i, col in enumerate(df_model.columns, 1):
    print(f'  {i:2d}. {col}')

## 5. Encodage des variables catégorielles

### Stratégie d'encodage

| Variable | Type d'encodage | Justification |
|----------|-----------------|---------------|
| `Region` | One-Hot | Pas d'ordre entre les régions |
| `TypeCulture` | One-Hot | Pas d'ordre entre les cultures |
| `TypeSol` | One-Hot | Pas d'ordre entre les types de sol |
| `Irrigation` | One-Hot | Pas d'ordre entre les systèmes d'irrigation |
| `Capteur` | One-Hot | Pas d'ordre entre les sources de relevé |
| `StadeCulture` | **Ordinal** | Ordre chronologique : Semis → Levée → Tallage → Floraison → Maturation → Récolte |

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Définition des colonnes par type
colonnes_onehot = ['Region', 'TypeCulture', 'TypeSol', 'Irrigation', 'Capteur']
colonne_ordinal = 'StadeCulture'

# Ordre chronologique du stade de culture
ordre_stades = ['Semis', 'Levée', 'Tallage', 'Floraison', 'Maturation', 'Récolte']

# Vérification que tous les stades sont bien dans l'ordre défini
stades_presents = df_model['StadeCulture'].unique()
stades_non_definis = [s for s in stades_presents if s not in ordre_stades]
if stades_non_definis:
    print(f"⚠️ Stades non définis dans l'ordre : {stades_non_definis}")
else:
    print("✅ Tous les stades sont correctement définis dans l'ordre chronologique.")

# Création du préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), colonnes_onehot),
        ('ordinal', OrdinalEncoder(categories=[ordre_stades]), [colonne_ordinal])
    ],
    remainder='passthrough'  # Conserver telles quelles les colonnes non listées
)

print('✅ Préprocesseur configuré.')

## 6. Séparation Features (X) et Cible (y)

In [ ]:
# Séparation features / cible
X = df_model.drop(columns=['AnomalieLabel'])
y = df_model['AnomalieLabel']

print(f'Features (X) : {X.shape[0]:,} lignes × {X.shape[1]} colonnes')
print(f'Cible (y)    : {y.shape[0]:,} valeurs')
print(f'\nDistribution de la cible :')
print(y.value_counts().to_string())
print(f'\nRatio anomalies : {y.mean()*100:.2f}%')

In [ ]:
# Sauvegarde des données préparées pour la Phase 4
X.to_csv(DATA_DIR / 'X_features.csv', index=False)
y.to_csv(DATA_DIR / 'y_cible.csv', index=False)
print('✅ Features et cible sauvegardés pour la Phase 4.')

## ✅ Bilan de la Phase 3

| Étape | Statut | Détail |
|-------|:------:|--------|
| Variable temporelle | ✅ | `AgeCulture_jours` = DateObservation − DateMiseEnCulture |
| Variable métier | ✅ | `EcartRendement_t_ha` et `RatioRendement` |
| Suppression colonnes inutiles | ✅ | Identifiants et dates brutes retirés |
| Encodage One-Hot | ✅ | Region, TypeCulture, TypeSol, Irrigation, Capteur |
| Encodage Ordinal | ✅ | StadeCulture (ordre chronologique) |
| Séparation X / y | ✅ | Prêt pour la modélisation |

### 🔜 Prochaine étape : Phase 4 — Modélisation

Nous allons entraîner et comparer plusieurs modèles :
- Baseline : Régression Logistique
- Random Forest
- **XGBoost** (modèle cible)
- Optimisation des hyperparamètres avec Optuna

---
# 🤖 Phase 4 : Modélisation

### Objectif de cette phase

1. **Split Train/Test** stratifié (80/20)
2. **Standardiser** les variables numériques
3. **Rééquilibrer** les classes avec SMOTE
4. **Entraîner et comparer** plusieurs modèles (Régression Logistique → Random Forest → XGBoost)
5. **Optimiser** les hyperparamètres avec Optuna
6. **Tracker** les expériences avec MLflow

## 1. Train/Test Split stratifié

In [ ]:
from sklearn.model_selection import train_test_split

# Split stratifié pour préserver la proportion d'anomalies dans les deux ensembles
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Ensemble d\'entraînement : {X_train.shape[0]:,} observations')
print(f'Ensemble de test        : {X_test.shape[0]:,} observations')
print(f'\nProportion d\'anomalies dans y_train : {y_train.mean()*100:.2f}%')
print(f'Proportion d\'anomalies dans y_test  : {y_test.mean()*100:.2f}%')

## 2. Application du préprocesseur et standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Application du préprocesseur (OneHot + Ordinal) sur les données d'entraînement
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Récupération des noms de colonnes après encodage
# Colonnes OneHot
onehot_cols = list(preprocessor.named_transformers_['onehot'].get_feature_names_out(colonnes_onehot))
# Colonne Ordinal
ordinal_cols = [colonne_ordinal]
# Colonnes numériques restantes (passthrough)
colonnes_numeriques = [col for col in X.columns if col not in colonnes_onehot and col != colonne_ordinal]
all_cols = onehot_cols + ordinal_cols + colonnes_numeriques

print(f'Nombre de features après encodage : {len(all_cols)}')
print(f'  - OneHot  : {len(onehot_cols)}')
print(f'  - Ordinal : {len(ordinal_cols)}')
print(f'  - Numériques : {len(colonnes_numeriques)}')

In [ ]:
# Standardisation des variables numériques
# On identifie les colonnes numériques dans le tableau encodé
# Les dernières colonnes (passthrough) sont les variables numériques
idx_debut_num = len(onehot_cols) + len(ordinal_cols)

scaler = StandardScaler()
X_train_encoded[:, idx_debut_num:] = scaler.fit_transform(X_train_encoded[:, idx_debut_num:])
X_test_encoded[:, idx_debut_num:] = scaler.transform(X_test_encoded[:, idx_debut_num:])

print('✅ Données standardisées.')

## 3. Gestion du déséquilibre de classes — SMOTE

### Verification NaN avant SMOTE

Avant d'appliquer SMOTE, on s'assure qu'il ne reste aucune valeur manquante dans les features encodees. En effet, SMOTE n'accepte pas les NaN.

In [ ]:
# Verification finale : plus aucune valeur manquante avant SMOTE
import numpy as np
from sklearn.impute import SimpleImputer

nan_count = np.isnan(X_train_encoded).sum()
if nan_count > 0:
    print(f'{nan_count} NaN residuels detectes dans X_train_encoded !')
    # Identifier les colonnes concernees
    nan_cols = np.isnan(X_train_encoded).sum(axis=0)
    for idx_col in range(X_train_encoded.shape[1]):
        if nan_cols[idx_col] > 0:
            col_name = all_cols[idx_col] if idx_col < len(all_cols) else f'colonne_{idx_col}'
            print(f'  - {col_name}: {int(nan_cols[idx_col])} NaN -> imputation par la mediane')
    # Imputation par la mediane pour chaque colonne
    imp = SimpleImputer(strategy='median')
    X_train_encoded = imp.fit_transform(X_train_encoded)
    X_test_encoded = imp.transform(X_test_encoded)
    print('OK NaN residuels traites.')
else:
    print('OK Aucun NaN dans X_train_encoded.')


In [ ]:
from imblearn.over_sampling import SMOTE

print(f'Avant SMOTE — Entraînement :')
print(f'  Normal   (0) : {(y_train == 0).sum():,}')
print(f'  Anomalie (1) : {(y_train == 1).sum():,}')

# Application de SMOTE uniquement sur l'ensemble d'entraînement
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_encoded, y_train)

print(f'\nAprès SMOTE — Entraînement :')
print(f'  Normal   (0) : {(y_train_resampled == 0).sum():,}')
print(f'  Anomalie (1) : {(y_train_resampled == 1).sum():,}')
print(f'\n✅ Classes équilibrées par SMOTE.')

## 4. Modèle Baseline — Régression Logistique

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score

# Entraînement de la baseline
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_resampled, y_train_resampled)

# Prédictions
y_pred_lr = lr.predict(X_test_encoded)
y_proba_lr = lr.predict_proba(X_test_encoded)[:, 1]

# Métriques
print('=== RÉGRESSION LOGISTIQUE (Baseline) ===')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_lr):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_lr):.4f}')
print(f'\nRapport de classification :')
print(classification_report(y_test, y_pred_lr, target_names=['Normal', 'Anomalie']))

## 5. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entraînement Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, 
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_resampled, y_train_resampled)

# Prédictions
y_pred_rf = rf.predict(X_test_encoded)
y_proba_rf = rf.predict_proba(X_test_encoded)[:, 1]

# Métriques
print('=== RANDOM FOREST ===')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_rf):.4f}')
print(f'\nRapport de classification :')
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Anomalie']))

## 6. XGBoost — Modèle principal

In [ ]:
import xgboost as xgb

# Calcul du ratio de déséquilibre pour scale_pos_weight
ratio_neg_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Ratio Normal/Anomalie (scale_pos_weight) : {ratio_neg_pos:.2f}')

# Entraînement XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=ratio_neg_pos,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_resampled, y_train_resampled)

# Prédictions
y_pred_xgb = xgb_model.predict(X_test_encoded)
y_proba_xgb = xgb_model.predict_proba(X_test_encoded)[:, 1]

# Métriques
print('=== XGBOOST ===')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_xgb):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_xgb):.4f}')
print(f'\nRapport de classification :')
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Anomalie']))

### 🟢 Comparaison des 3 modèles

Les métriques clés à comparer :

| Modèle | ROC-AUC | Recall | F1-Score | Interprétation |
|--------|---------|--------|----------|----------------|
| Régression Logistique | - | - | - | Baseline simple |
| Random Forest | - | - | - | Robuste, bon sur données mixtes |
| **XGBoost** | - | - | - | 🏆 Meilleur compromis attendu |

**Pourquoi le Recall est notre métrique prioritaire :**
Dans le contexte agricole, **rater une anomalie** (faux négatif) a un coût bien plus élevé que **déclencher une fausse alerte** (faux positif). Un faux négatif signifie qu'une parcelle en difficulté n'est pas signalée → perte de rendement potentielle. Un faux positif signifie juste une vérification inutile.

## 7. Optimisation des hyperparamètres avec Optuna

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'scale_pos_weight': ratio_neg_pos,
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss',
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train_resampled, y_train_resampled, 
                             cv=3, scoring='f1', n_jobs=-1)
    return scores.mean()

# Création de l'étude Optuna
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)

# Lancement de l'optimisation (50 essais)
print('Optimisation en cours...')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n=== MEILLEURS HYPERPARAMÈTRES ===')
for key, value in study.best_params.items():
    print(f'  {key}: {value}')
print(f'\nMeilleur F1-score CV : {study.best_value:.4f}')

## 8. Entraînement du modèle final avec les meilleurs hyperparamètres

In [ ]:
# Meilleurs hyperparamètres trouvés par Optuna
best_params = study.best_params
best_params.update({
    'scale_pos_weight': ratio_neg_pos,
    'random_state': 42,
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'n_jobs': -1
})

# Entraînement du modèle optimisé
xgb_optimized = xgb.XGBClassifier(**best_params)
xgb_optimized.fit(X_train_resampled, y_train_resampled)

# Prédictions
y_pred_opt = xgb_optimized.predict(X_test_encoded)
y_proba_opt = xgb_optimized.predict_proba(X_test_encoded)[:, 1]

# Métriques finales
print('=== XGBOOST OPTIMISÉ (Optuna) ===')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba_opt):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_opt):.4f}')
print(f'F1-Score : {f1_score(y_test, y_pred_opt):.4f}')
print(f'\nRapport de classification :')
print(classification_report(y_test, y_pred_opt, target_names=['Normal', 'Anomalie']))

## 9. Tracking MLflow

In [ ]:
import mlflow
import mlflow.sklearn

# Configuration MLflow
mlflow.set_tracking_uri(f'sqlite:///{DATA_DIR.parent.absolute()}/mlruns.db')
mlflow.set_experiment('CISIA_Agri_Anomalies')

with mlflow.start_run(run_name='XGBoost_Optuna'):
    # Log des hyperparamètres
    mlflow.log_params(best_params)
    
    # Log des métriques
    mlflow.log_metric('roc_auc', roc_auc_score(y_test, y_proba_opt))
    mlflow.log_metric('recall', recall_score(y_test, y_pred_opt))
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred_opt))
    
    # Log du modèle
    mlflow.sklearn.log_model(xgb_optimized, 'model')
    
    print('✅ Expérience MLflow enregistrée.')
    print(f'   URI : {mlflow.get_tracking_uri()}')

In [ ]:
# Sauvegarde du modèle pour utilisation ultérieure (API)
import joblib

# Sauvegarde du modèle XGBoost optimisé
joblib.dump(xgb_optimized, DATA_DIR / 'modele_xgboost_optuna.pkl')
# Sauvegarde du préprocesseur et du scaler
joblib.dump(preprocessor, DATA_DIR / 'preprocessor.pkl')
joblib.dump(scaler, DATA_DIR / 'scaler.pkl')

print('✅ Modèle et préprocesseur sauvegardés.')

## ✅ Bilan de la Phase 4

| Étape | Statut | Détail |
|-------|:------:|--------|
| Train/Test Split | ✅ | 80/20 stratifié |
| Encodage + Standardisation | ✅ | OneHot, Ordinal, StandardScaler |
| SMOTE | ✅ | Suréchantillonnage de la classe minoritaire |
| Régression Logistique | ✅ | Baseline |
| Random Forest | ✅ | Modèle de comparaison |
| XGBoost | ✅ | Modèle principal |
| Optuna | ✅ | 50 essais, optimisation bayésienne |
| XGBoost optimisé | ✅ | Meilleurs hyperparamètres appliqués |
| MLflow | ✅ | Tracking des paramètres et métriques |
| Sauvegarde modèle | ✅ | Modèle, préprocesseur et scaler sauvegardés |

### 🔜 Prochaine étape : Phase 5 — Évaluation finale & Architecture cible

---
# 📊 Phase 5 : Évaluation, Interprétabilité & Architecture cible

### Objectif de cette phase

1. **Comparer visuellement** les performances des modèles
2. **Analyser** la matrice de confusion du modèle final
3. **Expliquer** les prédictions avec SHAP
4. **Proposer** une architecture cible pour l'intégration en production

## 1. Courbes ROC comparées

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(10, 8))

# Courbes ROC des 4 modèles
RocCurveDisplay.from_predictions(y_test, y_proba_lr, ax=ax, name='Régression Logistique', color='blue', linestyle='--')
RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=ax, name='Random Forest', color='green', linestyle='-.')
RocCurveDisplay.from_predictions(y_test, y_proba_xgb, ax=ax, name='XGBoost', color='orange', linestyle=':')
RocCurveDisplay.from_predictions(y_test, y_proba_opt, ax=ax, name='XGBoost Optimisé', color='red', linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Aléatoire')
ax.set_title('Courbes ROC — Comparaison des modèles', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

### 🟢 Interprétation des courbes ROC

La courbe ROC (Receiver Operating Characteristic) montre le compromis entre :
- **Taux de vrais positifs** (Recall) — plus c'est haut, mieux c'est
- **Taux de faux positifs** — plus c'est bas, mieux c'est

L'AUC (Area Under Curve) mesure la capacité du modèle à discriminer les deux classes :
- AUC = 1.0 : discrimination parfaite
- AUC = 0.5 : modèle aléatoire
- **Plus la courbe est proche du coin supérieur gauche, meilleur est le modèle.**

👉 Le modèle XGBoost optimisé devrait avoir l'AUC la plus élevée.

## 2. Matrice de confusion du modèle final

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opt, 
    display_labels=['Normal', 'Anomalie'],
    cmap='Blues', ax=ax, colorbar=True
)
ax.set_title('Matrice de confusion — XGBoost Optimisé', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🟢 Interprétation de la matrice de confusion

La matrice de confusion se lit comme suit :

| | Prédit Normal | Prédit Anomalie |
|---|---|---|
| **Réel Normal** | ✅ Vrai Négatif (VN) | ⚠️ Faux Positif (FP) |
| **Réel Anomalie** | 🔴 Faux Négatif (FN) | ✅ Vrai Positif (VP) |

Dans notre contexte agricole :
- **Faux Négatifs (FN)** = Anomalies non détectées → **coût élevé** (perte de rendement)
- **Faux Positifs (FP)** = Fausses alertes → **coût faible** (vérification inutile)

👉 L'objectif est de **minimiser les FN** quitte à accepter quelques FP supplémentaires.

## 3. Feature Importance — Quels facteurs déclenchent les anomalies ?

In [ ]:
# Importance des features selon XGBoost
feature_importance = pd.DataFrame({
    'Feature': all_cols,
    'Importance': xgb_optimized.feature_importances_
}).sort_values('Importance', ascending=False)

# Top 15 features
top_features = feature_importance.head(15)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(top_features)))
bars = ax.barh(range(len(top_features)), top_features['Importance'].values[::-1], 
               color=colors[::-1], edgecolor='white')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values[::-1], fontsize=10)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Top 15 — Importance des features (XGBoost)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 🟢 Interprétation de l'importance des features

Ce graphique révèle les variables qui influencent le plus la décision du modèle :

- Les **variables métier** (EcartRendement, RatioRendement, AgeCulture) devraient figurer parmi les plus importantes — cela valide notre feature engineering
- Le **NDVI** est un indicateur agronomique clé de la santé végétative
- Les **types de culture et de sol** apportent un contexte important
- Si une variable créée manuellement arrive en tête, cela justifie le travail de feature engineering

👉 Ce graphique est essentiel pour **expliquer le modèle à la coopérative** en termes compréhensibles.

## 4. Explicabilité avec SHAP

In [ ]:
import shap

# Création de l'explainer SHAP (TreeExplainer optimisé pour XGBoost)
explainer = shap.TreeExplainer(xgb_optimized)

# Calcul des valeurs SHAP sur un échantillon du test (pour des raisons de performance)
X_test_sample = X_test_encoded[:500]
shap_values = explainer.shap_values(X_test_sample)

print('✅ Valeurs SHAP calculées.')

In [ ]:
# Summary plot — Impact global des features
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, feature_names=all_cols, 
                  show=False, max_display=15)
ax.set_title('SHAP Summary Plot — Impact des features sur la prédiction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🟢 Interprétation du SHAP Summary Plot

Le graphique SHAP est plus riche que la simple feature importance car il montre :
- **L'ampleur de l'impact** (axe horizontal) : à quel point la feature fait bouger la prédiction
- **La direction de l'impact** (couleur) : rouge = valeur élevée de la feature, bleu = valeur faible
- **La dispersion** : une feature peut avoir un impact positif dans certains cas et négatif dans d'autres

Exemple de lecture :
- Si `NDVI` apparaît en haut avec des points rouges à gauche (impact négatif) → un NDVI élevé **diminue** la probabilité d'anomalie (logique : bonne santé végétative)
- Si `EcartRendement_t_ha` a des points rouges à droite → un écart de rendement positif **augmente** la probabilité d'anomalie

👉 SHAP permet de répondre à la question : **pourquoi le modèle a prédit une anomalie sur cette parcelle ?**

## 5. Synthèse des performances

In [ ]:
# Tableau comparatif final
resultats = pd.DataFrame([
    ['Régression Logistique', roc_auc_score(y_test, y_proba_lr), 
     recall_score(y_test, y_pred_lr), f1_score(y_test, y_pred_lr)],
    ['Random Forest', roc_auc_score(y_test, y_proba_rf), 
     recall_score(y_test, y_pred_rf), f1_score(y_test, y_pred_rf)],
    ['XGBoost', roc_auc_score(y_test, y_proba_xgb), 
     recall_score(y_test, y_pred_xgb), f1_score(y_test, y_pred_xgb)],
    ['XGBoost Optimisé', roc_auc_score(y_test, y_proba_opt), 
     recall_score(y_test, y_pred_opt), f1_score(y_test, y_pred_opt)]
], columns=['Modèle', 'ROC-AUC', 'Recall', 'F1-Score'])

resultats = resultats.round(4)
display(resultats.style.highlight_max(subset=['ROC-AUC', 'Recall', 'F1-Score'], color='#90EE90'))

### 🟢 Synthèse et choix final

Le modèle retenu est **XGBoost optimisé par Optuna** car il offre le meilleur compromis entre :
- **Recall** : capacité à détecter les anomalies (priorité métier)
- **F1-Score** : équilibre précision/rappel
- **ROC-AUC** : pouvoir discriminant global

**Métrique cible pour la soutenance :** Recall ≥ 0.85 et F1 ≥ 0.80 (seuils indicatifs, à ajuster selon les résultats réels).

## 6. Architecture cible — Déploiement en production (Compétences C6, C7)

### Contexte

La coopérative a besoin d'une solution **opérationnelle et intégrable** dans ses outils existants. Le modèle entraîné doit pouvoir être :
1. **Appelé à la demande** via une API pour analyser une parcelle spécifique
2. **Exécuté en batch** pour l'analyse périodique de toutes les parcelles
3. **Mis à jour** avec de nouvelles données (réentraînement saisonnier)

### Architecture proposée

```
┌─────────────────────────────────────────────────────────┐
│                    COOPÉRATIVE AGRICOLE                  │
│  ┌──────────┐  ┌──────────┐  ┌──────────────────────┐   │
│  │ Outils   │  │ Dashboard│  │ Système d'alerte     │   │
│  │ existants│  │ Streamlit│  │ (email, SMS)         │   │
│  └────┬─────┘  └────┬─────┘  └──────────┬───────────┘   │
│       │             │                   │               │
└───────┼─────────────┼───────────────────┼───────────────┘
        │             │                   │
        └─────────────┼───────────────────┘
                      │  HTTP REST
              ┌───────▼────────┐
              │   API FastAPI  │
              │  /predict      │  ← Validation Pydantic
              │  /health       │
              │  /batch        │
              └───────┬────────┘
                      │
              ┌───────▼────────┐
              │ Modèle XGBoost │
              │ (MLflow)       │
              └───────┬────────┘
                      │
              ┌───────▼────────┐
              │  Préprocesseur │
              │  + Scaler      │
              └────────────────┘
```

### Composants techniques

| Composant | Technologie | Rôle |
|-----------|-------------|------|
| API REST | FastAPI + Uvicorn | Servir les prédictions via HTTP |
| Validation | Pydantic | Schémas stricts pour les données d'entrée |
| Modèle | XGBoost (joblib) | Inférence de classification binaire |
| Tracking | MLflow | Versionnement et registry du modèle |
| Conteneurisation | Docker | Reproductibilité de l'environnement |
| Orchestration | Docker Compose | API + MLflow + Dashboard |
| Dashboard | Streamlit (optionnel) | Visualisation des prédictions en temps réel |

### Endpoint principal : POST /predict

**Entrée :**
```json
{
  "temperature": 22.5,
  "humidite": 65.0,
  "pluviometrie_mm": 3.2,
  "ndvi": 0.68,
  "capteur": "Drone",
  "stade_culture": "Floraison",
  "rendement_estime": 7.8,
  "rendement_moyen_zone": 8.1,
  "type_culture": "Blé",
  "type_sol": "Limoneux",
  "region": "Occitanie",
  "irrigation": "Goutte-à-goutte",
  "surface_ha": 12.5,
  "age_culture_jours": 95
}
```

**Sortie :**
```json
{
  "anomalie_predite": 0,
  "probabilite_anomalie": 0.12,
  "seuil_alerte": 0.5,
  "facteurs_principaux": [
    {"feature": "ndvi", "contribution": 0.68},
    {"feature": "ecart_rendement", "contribution": 0.22}
  ]
}
```

### Pipeline CI/CD proposé

```
┌─────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  Push   │───▶│  Tests   │───▶│  Build   │───▶│  Deploy  │
│  GitHub │    │  Pytest  │    │  Docker  │    │  Serveur │
└─────────┘    └──────────┘    └──────────┘    └──────────┘
                     │               │               │
                     ▼               ▼               ▼
               Ruff (lint)    Docker Compose   Monitoring
               pytest-cov     MLflow Registry  Prometheus
```

### Amélioration continue (C9)

| Mécanisme | Description |
|-----------|-------------|
| **PSI Drift** | Population Stability Index — détection de dérive entre données d'entraînement et production |
| **Réentraînement saisonnier** | Mise à jour du modèle à chaque fin de campagne culturale |
| **Feedback loop** | Collecte des faux positifs/négatifs signalés par les exploitants |
| **A/B testing** | Comparaison du modèle actuel vs nouveau modèle avant déploiement |

Ces éléments répondent aux compétences C6 (Implémentation), C7 (Architecture cible) et C9 (Amélioration continue).

## ✅ Bilan de la Phase 5

| Étape | Statut | Détail |
|-------|:------:|--------|
| Courbes ROC comparées | ✅ | 4 modèles superposés |
| Matrice de confusion | ✅ | XGBoost optimisé |
| Feature Importance | ✅ | Top 15 features XGBoost |
| SHAP Summary Plot | ✅ | Explicabilité post-hoc |
| Tableau comparatif | ✅ | Synthèse des métriques |
| Architecture cible | ✅ | Schéma API + Docker + CI/CD |
| Amélioration continue | ✅ | PSI, réentraînement, feedback loop |

---
# 🎯 Conclusion générale

## Résumé du projet

Ce projet a permis de concevoir un système de **détection d'anomalies parcellaires** basé sur du Machine Learning, en suivant une démarche rigoureuse en 5 phases :

| Phase | Objectif | Résultat |
|:-----:|----------|----------|
| 1 | Ingestion & RGPD | Données fusionnées, données personnelles supprimées |
| 2 | Nettoyage & EDA | Valeurs aberrantes corrigées, valeurs manquantes imputées |
| 3 | Feature Engineering | Variables métier créées, catégorielles encodées |
| 4 | Modélisation | XGBoost optimisé par Optuna, suivi MLflow |
| 5 | Évaluation & Architecture | Métriques, SHAP, schéma d'intégration |

## Points clés pour la soutenance

1. **Choix du Recall** comme métrique prioritaire — justifié par le coût des faux négatifs
2. **XGBoost** comme modèle final — performant sur données tabulaires hétérogènes
3. **Feature engineering métier** — écart de rendement et âge de culture déterminants
4. **Architecture API** — solution opérationnelle et intégrable (FastAPI + Docker)
5. **Conformité RGPD** — suppression des données personnelles dès la Phase 1
6. **Biais identifiés** — géographiques, culturaux, instrumentaux (capteurs)

## Limites et perspectives

- **Données synthétiques** : les données fournies sont simulées, un déploiement réel nécessiterait une validation sur données terrain
- **Saisonnalité** : le modèle devra être réentraîné à chaque campagne pour rester pertinent
- **Nouvelles sources** : l'intégration d'images satellite et de prévisions météo enrichirait le modèle
- **Explicabilité** : SHAP répond au besoin d'explication, mais un dashboard métier (Streamlit) serait un plus pour la coopérative

---

*Notebook réalisé dans le cadre de la certification CISIA — Juin 2026*
